# BioSkills Lab — Chapter 11
## scVI and Single-Cell Representations
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> **GPU recommended** for faster scVI training (~2 min GPU vs ~15 min CPU)

> **Note:** The full experiment in the lesson uses large multi-donor single-cell datasets that require 32GB+ RAM. This notebook uses the PBMC 3k dataset (~2,700 cells) which runs on **free-tier Google Colab**. To reproduce experiments with larger datasets (100k+ cells), run on a local machine or compute cluster.

### What you will do
Apply **scVI** to the classic PBMC 3k dataset and compare it to the standard PCA pipeline.

**Why single-cell is different from bulk RNA-seq (Chapters 7-10):**
- Each row is an individual cell, not a patient sample
- Counts are sparse — most genes in most cells are zero
- Technical noise is higher: dropouts, PCR bias, library size variation
- Experiments often span multiple donors/batches that need harmonisation

**The key question:** Does scVI produce better cell clusters than standard PCA normalisation?

### Install dependencies
`scvi-tools` includes scVI and related models. `scanpy` is the standard single-cell analysis toolkit.

In [ ]:
!pip install -q scvi-tools scanpy anndata leidenalg

In [ ]:
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
print(f'scanpy: {sc.__version__}')
print(f'scvi-tools: {scvi.__version__}')

### Load and QC the PBMC 3k dataset

~2,700 peripheral blood mononuclear cells from a healthy donor (10x Chromium).

**QC steps:** Remove cells with <200 genes (empty droplets), genes in <3 cells (noise), and cells with >5% mitochondrial counts (dead/stressed cells).

**What to expect:** ~2,600 cells pass QC. Contains T cells, B cells, NK cells, monocytes, dendritic cells.

In [ ]:
adata = sc.datasets.pbmc3k()
print(f'Raw: {adata.shape[0]} cells, {adata.shape[1]} genes')

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
adata = adata[adata.obs.pct_counts_mt < 5].copy()
print(f'After QC: {adata.shape[0]} cells, {adata.shape[1]} genes')

### Select highly variable genes and store raw counts

We keep the 2000 most variable genes. **Critical:** raw counts are saved before normalisation — scVI requires raw integer counts. Passing normalised values to scVI gives wrong results.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor='seurat_v3')
adata.layers['counts'] = adata.X.copy()
print(f'HVGs selected: {adata.shape[1]}')
print('Raw counts saved in adata.layers["counts"]')

### Train scVI

scVI models each gene's count distribution as a **negative binomial** — appropriate for overdispersed count data.

- `n_latent=20`: 20-dimensional bottleneck (the learned cell representation)
- `n_layers=2`: 2 hidden layers
- `early_stopping=True`: stops when validation loss plateaus

**What to expect:** ~2 min GPU, ~15 min CPU.

In [ ]:
scvi.model.SCVI.setup_anndata(adata, layer='counts')
model = scvi.model.SCVI(adata, n_latent=20, n_layers=2)
model.train(max_epochs=200, early_stopping=True)
adata.obsm['X_scVI'] = model.get_latent_representation()
print(f'scVI complete. Latent shape: {adata.obsm["X_scVI"].shape}')

### Cluster and UMAP from scVI latent space

Nearest-neighbour graph in the 20D scVI space, then Leiden clustering and UMAP. **Expect** 6-10 clusters corresponding to major PBMC cell types.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI')
sc.tl.leiden(adata, resolution=0.5)
sc.tl.umap(adata)
sc.pl.umap(adata, color=['leiden'], title='scVI — Leiden Clusters',
           legend_loc='on data', legend_fontsize=8)

### Standard PCA pipeline for comparison

Classical scanpy pipeline: normalise -> log -> HVGs -> scale -> PCA -> neighbours -> Leiden -> UMAP. Direct comparison on the same cells.

In [ ]:
adata_pca = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata_pca, min_genes=200)
sc.pp.filter_genes(adata_pca, min_cells=3)
adata_pca.var['mt'] = adata_pca.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata_pca, qc_vars=['mt'], inplace=True)
adata_pca = adata_pca[adata_pca.obs.pct_counts_mt < 5].copy()
sc.pp.normalize_total(adata_pca, target_sum=1e4)
sc.pp.log1p(adata_pca)
sc.pp.highly_variable_genes(adata_pca, n_top_genes=2000, subset=True)
sc.pp.scale(adata_pca)
sc.tl.pca(adata_pca)
sc.pp.neighbors(adata_pca, n_pcs=40)
sc.tl.leiden(adata_pca, resolution=0.5, key_added='leiden_pca')
sc.tl.umap(adata_pca)
print('PCA pipeline complete')

### Side-by-side comparison

**How to interpret:** scVI clusters tend to be rounder and more compact. On single-donor PBMC 3k the difference is modest — scVI's advantage is most visible on **multi-donor datasets** where batch effects need correction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata,     color='leiden',     ax=axes[0], show=False, title='scVI clusters',
           legend_loc='on data', legend_fontsize=8)
sc.pl.umap(adata_pca, color='leiden_pca', ax=axes[1], show=False, title='PCA clusters',
           legend_loc='on data', legend_fontsize=8)
plt.suptitle('scVI vs PCA — PBMC 3k Cell Clustering', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
print('\nIn the Capstone you will extend this comparison to a multi-donor dataset and add Geneformer.')